<a href="https://colab.research.google.com/github/lesyeuxdehillarie/Predicting-Cost-Sharing-Requirements-in-Federal-Grant-Opportunities-Using-Machine-Learning/blob/main/JULIE_LANYERO_QUESTION_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **(a) Design a complete preprocessing pipeline (10 marks)**
Data preprocessing is the process of cleaning, organizing, and transforming raw data into a suitable format for machine learning.
1: Data collection and integration
Combine datasets from the different schools into one dataset. Check that the columns have the same meaning, even if they have different names or formats. For example, Student_ID and StudentNumber may refer to the same variable.

2: Data inspection
Use Python libraries such as Pandas and NumPy to examine the dataset. Check the number of records, column names, data types, missing values, duplicates, and unusual exam marks.

3: Handle missing values
Identify missing values in attendance and test scores. Fill missing attendance values using the median or mean where appropriate. For test scores, use the median or a suitable statistical imputation method. If many values are missing, investigate the cause before deciding whether to remove the affected records.

4: Remove duplicate records
Identify duplicate student records using Student ID and other relevant information. Remove confirmed duplicates while preserving legitimate records, such as a student's results from different terms.

5: Detect and handle outliers
Use the Interquartile Range (IQR) method or box plots to identify unusual exam marks. Check whether the marks are data-entry errors or genuine results. Correct errors where possible, and consider capping extreme values or retaining valid observations.

6: Standardize data formats
Convert dates, attendance percentages, school names, and examination scores into consistent formats. For example, change attendance values recorded as 85% and 0.85 into a common representation.

7: Encode categorical variables
Convert categorical data, such as gender, school, and class, into numerical values using techniques such as one-hot encoding or label encoding, depending on the variable and machine learning algorithm.

8: Scale numerical features
Use standardization or normalization for numerical variables such as attendance and study hours when required by the selected machine learning algorithm. Standardization converts values to a common scale based on their mean and standard deviation.

9: Feature selection and target definition
Select relevant features such as attendance, previous test scores, and study hours. Define the target variable, for example, final examination score or pass/fail status. Remove irrelevant features and variables that could cause data leakage.

10: Split the dataset and validate the pipeline
Divide the dataset into training and testing sets, for example, 80% for training and 20% for testing. Fit imputation, encoding, and scaling procedures using the training data only, then apply them to the test data. Evaluate the processed data before training the model.

#1. **Import libraries**

In [ ]:
import pandas as pd
import numpy as np

from google.colab import files
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully!")

Libraries imported successfully!


# **2.Upload the three CSV files**

In [ ]:
uploaded = files.upload()

Saving School_C.csv to School_C.csv
Saving School_A.csv to School_A.csv
Saving School_B.csv to School_B.csv


# **3.Load and inspect the datasets**

In [ ]:
school_a = pd.read_csv("School_A.csv")
school_b = pd.read_csv("School_B.csv")
school_c = pd.read_csv("School_C.csv")

print("School A:", school_a.shape)
display(school_a.head())

print("School B:", school_b.shape)
display(school_b.head())

print("School C:", school_c.shape)
display(school_c.head())

School A: (43, 8)


,Student_ID,School,Age,Gender,Attendance_Percent,Test_Score_100,Exam_Mark_100,Term
0,A001,School A,19,Male,73.0,27.0,21,2026-T1
1,A002,School A,19,Male,91.0,32.0,38,2026-T1
2,A003,School A,19,Male,79.0,91.0,61,2026-T1
3,A004,School A,17,Female,NaN,32.0,44,2026-T1
4,A005,School A,19,Female,67.0,25.0,34,2026-T1


School B: (43, 8)


,learner_id,school_name,age_years,sex,attendance_rate,test_score_50,final_exam_100,term
0,B001,School B,18,M,0.60,14.0,29,Term 1 2026
1,B002,School B,14,F,0.86,38.0,69,Term 1 2026
2,B003,School B,18,M,0.93,44.5,61,Term 1 2026
3,B004,School B,17,F,NaN,36.0,66,Term 1 2026
4,B005,School B,16,M,0.68,25.0,49,Term 1 2026


School C: (43, 8)


,student_code,institution,age,gender,attendance,continuous_test,exam_result,academic_term
0,C001,School C,18,female,66%,73/100,60,T1-2026
1,C002,School C,14,female,59%,90/100,63,T1-2026
2,C003,School C,16,female,77%,73/100,52,T1-2026
3,C004,School C,17,male,NaN,79/100,65,T1-2026
4,C005,School C,18,male,68%,42/100,38,T1-2026


# **4.Standardize column names**

In [ ]:
# Standardize column names across the three schools

school_a = school_a.rename(columns={
    "Student_ID": "Student_ID",
    "School": "School",
    "Age": "Age",
    "Gender": "Gender",
    "Attendance_Percent": "Attendance_Percent",
    "Test_Score_100": "Test_Score_100",
    "Exam_Mark_100": "Exam_Mark_100",
    "Term": "Term"
})

school_b = school_b.rename(columns={
    "learner_id": "Student_ID",
    "school_name": "School",
    "age_years": "Age",
    "sex": "Gender",
    "attendance_rate": "Attendance_Percent",
    "test_score_50": "Test_Score_100",
    "final_exam_100": "Exam_Mark_100",
    "term": "Term"
})

school_c = school_c.rename(columns={
    "student_code": "Student_ID",
    "institution": "School",
    "age": "Age",
    "gender": "Gender",
    "attendance": "Attendance_Percent",
    "continuous_test": "Test_Score_100",
    "exam_result": "Exam_Mark_100",
    "academic_term": "Term"
})

print("School A columns:", school_a.columns.tolist())
print("School B columns:", school_b.columns.tolist())
print("School C columns:", school_c.columns.tolist())

School A columns: ['Student_ID', 'School', 'Age', 'Gender', 'Attendance_Percent', 'Test_Score_100', 'Exam_Mark_100', 'Term']
School B columns: ['Student_ID', 'School', 'Age', 'Gender', 'Attendance_Percent', 'Test_Score_100', 'Exam_Mark_100', 'Term']
School C columns: ['Student_ID', 'School', 'Age', 'Gender', 'Attendance_Percent', 'Test_Score_100', 'Exam_Mark_100', 'Term']


# **5.Combine the datasets**

In [ ]:
df = pd.concat(
    [school_a, school_b, school_c],
    ignore_index=True
)

print("Combined dataset shape:", df.shape)
display(df.head())

Combined dataset shape: (129, 8)


,Student_ID,School,Age,Gender,Attendance_Percent,Test_Score_100,Exam_Mark_100,Term
0,A001,School A,19,Male,73.0,27.0,21,2026-T1
1,A002,School A,19,Male,91.0,32.0,38,2026-T1
2,A003,School A,19,Male,79.0,91.0,61,2026-T1
3,A004,School A,17,Female,NaN,32.0,44,2026-T1
4,A005,School A,19,Female,67.0,25.0,34,2026-T1


# **6.Check missing values**

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

Missing values in each column:
Student_ID             0
School                 0
Age                    0
Gender                 0
Attendance_Percent    12
Test_Score_100        12
Exam_Mark_100          0
Term                   0
dtype: int64

Total missing values: 24


# **7.Convert values into consistent formats**

In [ ]:
# Function to clean missing text values
def clean_text(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value.lower() in ["", "na", "n/a", "nan", "null", "none"]:
        return np.nan

    return value


# Convert values to numbers
def to_number(value):
    value = clean_text(value)

    if pd.isna(value):
        return np.nan

    try:
        return float(str(value).replace(",", ""))
    except ValueError:
        return np.nan


# Standardize gender
def clean_gender(value):
    value = clean_text(value)

    if pd.isna(value):
        return np.nan

    gender = value.lower()

    if gender in ["f", "female", "woman"]:
        return "Female"
    elif gender in ["m", "male", "man"]:
        return "Male"

    return value.title()


# Convert attendance to percentages
def clean_attendance(value, school_code):
    value = clean_text(value)

    if pd.isna(value):
        return np.nan

    if str(value).endswith("%"):
        return to_number(str(value)[:-1])

    number = to_number(value)

    if pd.isna(number):
        return np.nan

    if school_code == "B" and 0 <= number <= 1:
        return number * 100

    return number


# Convert test scores to a score out of 100
def clean_test_score(value, school_code):
    value = clean_text(value)

    if pd.isna(value):
        return np.nan

    if "/" in str(value):
        parts = str(value).split("/")

        if len(parts) == 2:
            numerator = to_number(parts[0])
            denominator = to_number(parts[1])

            if pd.notna(numerator) and pd.notna(denominator) and denominator > 0:
                return (numerator / denominator) * 100

        return np.nan

    number = to_number(value)

    if pd.isna(number):
        return np.nan

    if school_code == "B":
        return number * 2

    return number


# Apply cleaning to each school's dataset
school_a["Age"] = school_a["Age"].apply(to_number)
school_b["Age"] = school_b["Age"].apply(to_number)
school_c["Age"] = school_c["Age"].apply(to_number)

school_a["Gender"] = school_a["Gender"].apply(clean_gender)
school_b["Gender"] = school_b["Gender"].apply(clean_gender)
school_c["Gender"] = school_c["Gender"].apply(clean_gender)

school_a["Attendance_Percent"] = school_a["Attendance_Percent"].apply(
    lambda x: clean_attendance(x, "A")
)
school_b["Attendance_Percent"] = school_b["Attendance_Percent"].apply(
    lambda x: clean_attendance(x, "B")
)
school_c["Attendance_Percent"] = school_c["Attendance_Percent"].apply(
    lambda x: clean_attendance(x, "C")
)

school_a["Test_Score_100"] = school_a["Test_Score_100"].apply(
    lambda x: clean_test_score(x, "A")
)
school_b["Test_Score_100"] = school_b["Test_Score_100"].apply(
    lambda x: clean_test_score(x, "B")
)
school_c["Test_Score_100"] = school_c["Test_Score_100"].apply(
    lambda x: clean_test_score(x, "C")
)

for school in [school_a, school_b, school_c]:
    school["Exam_Mark_100"] = school["Exam_Mark_100"].apply(to_number)

# Combine the cleaned datasets again
df = pd.concat(
    [school_a, school_b, school_c],
    ignore_index=True
)

print("Data formats standardized successfully!")
display(df.head())

Data formats standardized successfully!


,Student_ID,School,Age,Gender,Attendance_Percent,Test_Score_100,Exam_Mark_100,Term
0,A001,School A,19.0,Male,73.0,27.0,21.0,2026-T1
1,A002,School A,19.0,Male,91.0,32.0,38.0,2026-T1
2,A003,School A,19.0,Male,79.0,91.0,61.0,2026-T1
3,A004,School A,17.0,Female,NaN,32.0,44.0,2026-T1
4,A005,School A,19.0,Female,67.0,25.0,34.0,2026-T1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **8.Check and flag invalid values**

In [ ]:
df["Invalid_Age"] = df["Age"].notna() & ~df["Age"].between(1, 120)

df["Invalid_Attendance"] = (
    df["Attendance_Percent"].notna()
    & ~df["Attendance_Percent"].between(0, 100)
)

df["Invalid_Test_Score"] = (
    df["Test_Score_100"].notna()
    & ~df["Test_Score_100"].between(0, 100)
)

df["Invalid_Exam_Mark"] = (
    df["Exam_Mark_100"].notna()
    & ~df["Exam_Mark_100"].between(0, 100)
)

print("Invalid values found:")
print(df[[
    "Invalid_Age",
    "Invalid_Attendance",
    "Invalid_Test_Score",
    "Invalid_Exam_Mark"
]].sum())

Invalid values found:
Invalid_Age           0
Invalid_Attendance    0
Invalid_Test_Score    0
Invalid_Exam_Mark     9
dtype: int64


# **9.Handle invalid values and check missing data**

In [ ]:
df.loc[df["Invalid_Age"], "Age"] = np.nan
df.loc[df["Invalid_Attendance"], "Attendance_Percent"] = np.nan
df.loc[df["Invalid_Test_Score"], "Test_Score_100"] = np.nan
df.loc[df["Invalid_Exam_Mark"], "Exam_Mark_100"] = np.nan

print("Missing values after validation:")
print(df.isnull().sum())

Missing values after validation:
Student_ID             0
School                 0
Age                    0
Gender                 0
Attendance_Percent    12
Test_Score_100        12
Exam_Mark_100          9
Term                   0
Invalid_Age            0
Invalid_Attendance     0
Invalid_Test_Score     0
Invalid_Exam_Mark      0
dtype: int64


# **10.Handle missing values**


In [ ]:
print("Number of exact duplicate rows:", df.duplicated().sum())

Number of exact duplicate rows: 9


# **10.Remove Duplicates**

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

print("Dataset shape after removing duplicates:", df.shape)
print("Remaining exact duplicates:", df.duplicated().sum())

Dataset shape after removing duplicates: (120, 12)
Remaining exact duplicates: 0


# **12.Detect outliers in exam marks**

In [ ]:
Q1 = df["Exam_Mark_100"].quantile(0.25)
Q3 = df["Exam_Mark_100"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["Exam_Mark_100"] < lower_bound) |
    (df["Exam_Mark_100"] > upper_bound)
]

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of potential outliers:", len(outliers))

display(outliers[["Student_ID", "School", "Exam_Mark_100"]].head(20))

Lower bound: 13.0
Upper bound: 93.0
Number of potential outliers: 3


,Student_ID,School,Exam_Mark_100
30,A031,School A,8.0
70,B031,School B,8.0
110,C031,School C,8.0


# **13.Mark the potential outliers**

In [ ]:
df["Exam_Outlier"] = (
    (df["Exam_Mark_100"] < lower_bound) |
    (df["Exam_Mark_100"] > upper_bound)
)

print("Outlier flag counts:")
print(df["Exam_Outlier"].value_counts())

display(df.head())

Outlier flag counts:
Exam_Outlier
False    117
True       3
Name: count, dtype: int64


,Student_ID,School,Age,Gender,Attendance_Percent,Test_Score_100,Exam_Mark_100,Term,Invalid_Age,Invalid_Attendance,Invalid_Test_Score,Invalid_Exam_Mark,Exam_Outlier
0,A001,School A,19.0,Male,73.0,27.0,21.0,2026-T1,False,False,False,False,False
1,A002,School A,19.0,Male,91.0,32.0,38.0,2026-T1,False,False,False,False,False
2,A003,School A,19.0,Male,79.0,91.0,61.0,2026-T1,False,False,False,False,False
3,A004,School A,17.0,Female,NaN,32.0,44.0,2026-T1,False,False,False,False,False
4,A005,School A,19.0,Female,67.0,25.0,34.0,2026-T1,False,False,False,False,False


# **14.Check the cleaned dataset**

In [ ]:
print("Dataset size:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

display(df.head(10))

Dataset size: (120, 13)

Column names:
['Student_ID', 'School', 'Age', 'Gender', 'Attendance_Percent', 'Test_Score_100', 'Exam_Mark_100', 'Term', 'Invalid_Age', 'Invalid_Attendance', 'Invalid_Test_Score', 'Invalid_Exam_Mark', 'Exam_Outlier']

Data types:
Student_ID             object
School                 object
Age                   float64
Gender                 object
Attendance_Percent    float64
Test_Score_100        float64
Exam_Mark_100         float64
Term                   object
Invalid_Age              bool
Invalid_Attendance       bool
Invalid_Test_Score       bool
Invalid_Exam_Mark        bool
Exam_Outlier             bool
dtype: object

Missing values:
Student_ID             0
School                 0
Age                    0
Gender                 0
Attendance_Percent    12
Test_Score_100        12
Exam_Mark_100          9
Term                   0
Invalid_Age            0
Invalid_Attendance     0
Invalid_Test_Score     0
Invalid_Exam_Mark      0
Exam_Outlier           0

,Student_ID,School,Age,Gender,Attendance_Percent,Test_Score_100,Exam_Mark_100,Term,Invalid_Age,Invalid_Attendance,Invalid_Test_Score,Invalid_Exam_Mark,Exam_Outlier
0,A001,School A,19.0,Male,73.0,27.0,21.0,2026-T1,False,False,False,False,False
1,A002,School A,19.0,Male,91.0,32.0,38.0,2026-T1,False,False,False,False,False
2,A003,School A,19.0,Male,79.0,91.0,61.0,2026-T1,False,False,False,False,False
3,A004,School A,17.0,Female,NaN,32.0,44.0,2026-T1,False,False,False,False,False
4,A005,School A,19.0,Female,67.0,25.0,34.0,2026-T1,False,False,False,False,False
5,A006,School A,17.0,Female,99.0,51.0,57.0,2026-T1,False,False,False,False,False
6,A007,School A,14.0,Male,73.0,NaN,38.0,2026-T1,False,False,False,False,False
7,A008,School A,17.0,Male,93.0,44.0,46.0,2026-T1,False,False,False,False,False
8,A009,School A,19.0,Male,65.0,83.0,NaN,2026-T1,False,False,False,True,False
9,A010,School A,19.0,Male,55.0,77.0,62.0,2026-T1,False,False,False,False,False


# **15.Prepare the features and target variable**
- Target (y): Exam_Mark_100
- Features (X): Age, gender, attendance, test score, school, and term.
We will exclude Student_ID because it is an identifier, and Exam_Outlier because it was created using the exam marks we are trying to predict.

In [ ]:
# Features used to predict the exam mark
feature_columns = [
    "Age",
    "Gender",
    "Attendance_Percent",
    "Test_Score_100",
    "School",
    "Term"
]

X = df[feature_columns].copy()
y = df["Exam_Mark_100"].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())

Features shape: (120, 6)
Target shape: (120,)


,Age,Gender,Attendance_Percent,Test_Score_100,School,Term
0,19.0,Male,73.0,27.0,School A,2026-T1
1,19.0,Male,91.0,32.0,School A,2026-T1
2,19.0,Male,79.0,91.0,School A,2026-T1
3,17.0,Female,NaN,32.0,School A,2026-T1
4,19.0,Female,67.0,25.0,School A,2026-T1


,Exam_Mark_100
0,21.0
1,38.0
2,61.0
3,44.0
4,34.0


# ** 16.Split the data into training and testing sets**
Because a student may have multiple records, we should try to keep each student's records in only one set. This helps prevent the model from learning from one record of a student and being tested on another record of the same student.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=df["Student_ID"])
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training percentage:", round(len(X_train) / len(df) * 100, 2), "%")
print("Testing percentage:", round(len(X_test) / len(df) * 100, 2), "%")

Training records: 84
Testing records: 36
Training percentage: 70.0 %
Testing percentage: 30.0 %


# **17.Check and remove missing exam marks**

In [ ]:

# Check and remove missing exam marks

print("Missing exam marks before cleaning:",
      df["Exam_Mark_100"].isna().sum())

df = df.dropna(subset=["Exam_Mark_100"]).copy()

# Prepare features and target
X = df[feature_columns].copy()
y = df["Exam_Mark_100"].copy()

# Split data into training and testing sets
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=df["Student_ID"])
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Missing target values in training:", y_train.isna().sum())
print("Missing target values in testing:", y_test.isna().sum())
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Cleaning and splitting completed!")

Missing exam marks before cleaning: 9
Missing target values in training: 0
Missing target values in testing: 0
Training records: 77
Testing records: 34
Cleaning and splitting completed!


# **18.Encode categories, scale numbers, and build the model**
This cell handles missing values and categorical variables within a machine-learning pipeline. It learns the preprocessing values from the training data only.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

numeric_features = [
    "Age",
    "Attendance_Percent",
    "Test_Score_100"
]

categorical_features = [
    "Gender",
    "School",
    "Term"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


# **19.Evaluate the model**

In [ ]:
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("Mean Absolute Error (MAE):", round(mae, 2))
print("Root Mean Squared Error (RMSE):", round(rmse, 2))
print("R-squared (R²):", round(r2, 3))

Mean Absolute Error (MAE): 10.92
Root Mean Squared Error (RMSE): 14.92
R-squared (R²): 0.282


MAE: The average absolute difference between predicted and actual exam marks. Lower is better.

RMSE: Similar to MAE, but penalizes larger errors more heavily. Lower is better.

R²: Indicates how well the model explains variation in exam marks. Values closer to 1 generally indicate a better fit; a negative value means it performs worse than a simple mean-based baseline on this test set.

# **(b) Explain how each preprocessing step improves data quality**
1. Data integration	Combines information from different schools into one consistent dataset.
2. Data inspection	Identifies errors, missing values, inconsistent formats, and unusual records.
3. Missing-value handling	Reduces incomplete data and helps the model learn from available information.
4. Duplicate removal	Prevents repeated records from giving certain students or outcomes excessive influence.
5. Outlier handling	Reduces the effect of incorrect extreme values while preserving genuine student performance differences.
6. Standardization of formats	Ensures values are interpreted consistently across schools.
7. Encoding categorical variables	Converts categorical information into a form that machine learning algorithms can process.
8. Feature scaling	Prevents features with large numerical ranges from dominating scale-sensitive algorithms.
9. Feature selection	Removes irrelevant or leaking information and focuses learning on useful predictors.
10. Train-test splitting	Allows an independent evaluation of how well the model predicts unseen student outcomes.

# **(c) Discuss the risks to AI decision-making if preprocessing is poorly performed**
1. Inaccurate predictions
Missing values, incorrect scores, and inconsistent records can cause the model to make wrong predictions about students' academic performance.
2. Biased decisions
If data from some schools or groups of students is incomplete or poorly represented, the model may produce unfair predictions. For example, students from schools with poor record-keeping may be incorrectly classified as low performers.
3. Unfair allocation of resources
Schools may use predictions to allocate scholarships, remedial lessons, or other educational support. Incorrect predictions could deny deserving students assistance or direct resources to students who need them less.
4. Overfitting and misleading results
Duplicate records or repeated student information may cause the model to memorize patterns rather than learn general relationships. If related records appear in both training and testing sets, model performance may appear better than it really is.
5. Loss of genuine information
Removing all outliers without investigation may eliminate students with exceptionally high or low marks. This can prevent the model from learning about genuine differences in student performance.
6. Poor generalization
If schools use different grading systems or attendance formats and these differences are not standardized, the model may perform poorly when predicting outcomes for students from a new school.
7. Data leakage
If information that would not be available at prediction time is included in training, the model may appear highly accurate during testing but fail in real-world use.
8. Loss of trust and ethical concerns
Unreliable predictions can damage confidence in AI systems, expose students to unfair treatment, and lead to decisions that negatively affect their educational opportunities.